# Preliminary report pipeline — fragment graphs + base GIN

Produces everything the preliminary report can have tonight:
1. **Table 1** — fragment statistics for BBBP, BACE, ESOL (scaffold splits)
2. **Fragment graphs** saved to Drive (nodes = BRICS fragments with text, edges = broken bonds)
3. **Base GIN** on the fragment graphs, 3 seeds, test ROC-AUC / RMSE
4. **Figure** — fragment-count histogram with the 12-fragment line

**Before running:** *Runtime → Change runtime type → L4 GPU* (T4 also fine; this part is small). Then *Run all*. Total time is roughly 10–20 minutes.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/Jenifer code 2026"      # folder with BBBP.csv, bace.csv, esol.csv
OUT  = f"{ROOT}/dl_project"                            # everything this notebook writes goes here
CSVS = {"BBBP": f"{ROOT}/BBBP.csv", "BACE": f"{ROOT}/bace.csv", "ESOL": f"{ROOT}/esol.csv"}

!pip install -q rdkit torch_geometric
import os, sys, random, json
os.makedirs(OUT, exist_ok=True); os.chdir(OUT); sys.path.insert(0, OUT)
import numpy as np, pandas as pd, torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(no GPU: Runtime -> Change runtime type)")

## 2. Fragmentation code (written into the project folder)

In [ ]:
%%writefile fragment_graphs.py
"""
BRICS fragment graphs for MoleculeNet (BBBP, BACE, ESOL), scaffold splits.

Each molecule becomes a graph whose NODES are BRICS fragments and whose EDGES are
the bonds BRICS broke. Every node carries text (fragment SMILES + functional-group
names), so each molecule is a small text-attributed graph that GSPELL can explain.

Two modes
    default   counter-ions and other extra components are removed (largest component kept)
    --ionic   BRICS + ionic decomposition: every component is kept; metal-heteroatom bonds
              written as covalent (e.g. CC(=O)O[Na]) are split into ions; each counter-ion /
              salt partner is one node, linked to the main molecule by an "ionic" edge

Usage
    python fragment_graphs.py --demo "O=C(Nc1ccccc1)c1ccccc1"   # inspect one molecule
    python fragment_graphs.py --demo "CCN(CC)CC.Cl" --ionic     # same, keeping the salt
    python fragment_graphs.py --stats                           # dataset statistics table (CPU, minutes)
    python fragment_graphs.py --build                           # also save PyG graphs to ./frag_graphs/

Requires: rdkit, numpy, pandas, deepchem (for scaffold splits); torch + torch_geometric only for --build.
"""

import argparse
import os

import numpy as np
import pandas as pd
from rdkit import Chem, RDLogger
from rdkit.Chem import BRICS, rdFingerprintGenerator, rdMolDescriptors
from rdkit.Chem.MolStandardize import rdMolStandardize

RDLogger.DisableLog("rdApp.*")

DATASETS = {
    # name: (DeepChem loader, task type)
    "BBBP": ("load_bbbp", "classification"),
    "BACE": ("load_bace_classification", "classification"),
    "ESOL": ("load_delaney", "regression"),
}
MAX_FRAGMENTS_FOR_EXACT_SHAPLEY = 12
FP_BITS = 512

# Readable functional-group names for node text. A group is assigned to a fragment
# only if every atom of the match lies inside that fragment.
FUNCTIONAL_GROUPS = {
    "carboxylic acid": "[CX3](=O)[OX2H1]",
    "carboxylate": "[CX3](=O)[O-]",
    "protonated amine (ammonium)": "[NX4+;!$(N~O)]",
    "ester": "[#6][CX3](=O)[OX2H0][#6]",
    "amide": "[NX3][CX3]=[OX1]",
    "ketone or aldehyde": "[#6][CX3](=O)[#1,#6]",
    "hydroxyl": "[OX2H][#6]",
    "ether": "[OD2]([#6])[#6]",
    "primary amine": "[NX3;H2;!$(NC=O)][#6]",
    "secondary amine": "[NX3;H1;!$(NC=O)]([#6])[#6]",
    "tertiary amine": "[NX3;H0;!$(NC=O);!$(N=*);!a]([#6])([#6])[#6]",
    "nitro": "[N+](=O)[O-]",
    "nitrile": "C#N",
    "sulfonamide": "S(=O)(=O)N",
    "sulfone or sulfonyl": "[SX4](=O)(=O)",
    "thioether": "[SX2]([#6])[#6]",
    "trifluoromethyl": "C(F)(F)F",
    "halogen": "[F,Cl,Br,I]",
    "phosphate or phosphonate": "P(=O)(O)",
    "benzene ring": "c1ccccc1",
    "aromatic nitrogen heterocycle": "[n]",
    "aromatic oxygen or sulfur heterocycle": "[o,s]",
    "alkene": "[CX3]=[CX3]",
    "carbonyl": "[CX3]=[OX1]",
}
# Drop generic labels when a more specific one already covers them.
SUPPRESS = {
    "carboxylic acid": {"hydroxyl", "carbonyl", "ketone or aldehyde"},
    "carboxylate": {"carbonyl", "ketone or aldehyde"},
    "ester": {"ether", "carbonyl", "ketone or aldehyde"},
    "amide": {"carbonyl", "ketone or aldehyde"},
    "ketone or aldehyde": {"carbonyl"},
    "sulfonamide": {"sulfone or sulfonyl"},
    "trifluoromethyl": {"halogen"},
}
ELEMENT_NAMES = {"N": "nitrogen", "O": "oxygen", "S": "sulfur", "C": "carbon", "P": "phosphorus"}
_FG_PATTERNS = {name: Chem.MolFromSmarts(s) for name, s in FUNCTIONAL_GROUPS.items()}
_MORGAN = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=FP_BITS)
_LARGEST = rdMolStandardize.LargestFragmentChooser()
_METALS = rdMolStandardize.MetalDisconnector()

# Readable names for common counter-ions and salt partners (canonical SMILES of the component).
ION_NAMES = {
    "[Cl-]": "chloride ion", "Cl": "chloride (HCl salt)", "[Br-]": "bromide ion", "Br": "bromide (HBr salt)",
    "[I-]": "iodide ion", "I": "iodide (HI salt)", "[F-]": "fluoride ion",
    "[Na+]": "sodium ion", "[K+]": "potassium ion", "[Li+]": "lithium ion", "[Ca+2]": "calcium ion",
    "[Mg+2]": "magnesium ion", "[Zn+2]": "zinc ion", "[NH4+]": "ammonium ion", "N": "ammonia",
    "O": "water (hydrate)",
    "O=S(=O)(O)O": "sulfuric acid (sulfate salt)", "O=S(=O)([O-])[O-]": "sulfate ion",
    "O=S(=O)([O-])O": "hydrogen sulfate ion",
    "CS(=O)(=O)O": "methanesulfonic acid (mesylate salt)", "CS(=O)(=O)[O-]": "mesylate ion",
    "Cc1ccc(S(=O)(=O)O)cc1": "p-toluenesulfonic acid (tosylate salt)",
    "O=P(O)(O)O": "phosphoric acid (phosphate salt)", "O=[N+]([O-])O": "nitric acid (nitrate salt)",
    "O=C(O)/C=C\\C(=O)O": "maleic acid (maleate salt)", "O=C(O)/C=C/C(=O)O": "fumaric acid (fumarate salt)",
    "O=C(O)C(=O)O": "oxalic acid (oxalate salt)", "O=C(O)CC(O)(CC(=O)O)C(=O)O": "citric acid (citrate salt)",
    "O=C(O)C(O)C(O)C(=O)O": "tartaric acid (tartrate salt)", "CC(=O)O": "acetic acid (acetate salt)",
}
# Basic nitrogens that an acid counter-ion would protonate (amines and aromatic N, not amides)
_BASIC_N = Chem.MolFromSmarts("[NX3;!$(N-C=[O,S,N]);!$(N-S(=O)=O);!$(N-a)]")
_AROM_N = Chem.MolFromSmarts("[nX2]")
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE, Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]


# ----------------------------------------------------------------------------- fragmentation

def prepare_mol(smiles, ionic=False):
    """Parse SMILES. Default: keep the largest component (drops counter-ions such as '.Cl').
    ionic=True: keep every component and split metal-heteroatom bonds into ions
    (RDKit MetalDisconnector; atom order is preserved).
    Returns (mol, changed) or (None, False) if the SMILES is invalid."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, False
    if ionic:
        n_before = len(Chem.GetMolFrags(mol))
        try:
            split = _METALS.Disconnect(mol)
            Chem.SanitizeMol(split)       # recompute rings/valences after bonds were removed
            mol = split
        except Exception:                 # keep the molecule as written rather than drop it,
            pass                          # so both modes contain the same molecules
        return mol, len(Chem.GetMolFrags(mol)) != n_before
    if len(Chem.GetMolFrags(mol)) > 1:
        return _LARGEST.choose(mol), True
    return mol, False


def brics_partition(mol, allowed_atoms=None):
    """Split a molecule at BRICS bonds, keeping atom indices.

    Unlike BRICS.BRICSDecompose (which returns a set of unique SMILES), this keeps
    duplicate fragments, maps every atom to its fragment, and records which bonds
    were broken, which is what you need to build fragment-level edges.

    Returns
        groups:       list of tuples of atom indices, one per fragment
        frag_of_atom: list, fragment index for each atom
        broken:       list of (atom_i, atom_j) for each broken bond
    """
    pairs = sorted({tuple(sorted(p)) for p, _ in BRICS.FindBRICSBonds(mol)})
    if allowed_atoms is not None:           # ionic mode: counter-ions stay whole
        pairs = [p for p in pairs if p[0] in allowed_atoms and p[1] in allowed_atoms]
    bond_ids = [mol.GetBondBetweenAtoms(i, j).GetIdx() for i, j in pairs]
    pieces = Chem.FragmentOnBonds(mol, bond_ids, addDummies=False) if bond_ids else mol
    groups = list(Chem.GetMolFrags(pieces, asMols=False, sanitizeFrags=False))
    frag_of_atom = [None] * mol.GetNumAtoms()
    for f, group in enumerate(groups):
        for a in group:
            frag_of_atom[a] = f
    return groups, frag_of_atom, pairs


LABEL_SCHEME = "extended"   # "extended": functional_groups.label_fragment (curated + rings + Ertl)
                            # "basic": the original short SMARTS list below


def label_version():
    if LABEL_SCHEME == "extended":
        from functional_groups import LABEL_VERSION
        return LABEL_VERSION
    return "v1"


def functional_groups(mol, group):
    """Readable labels for one fragment, matched in the parent molecule."""
    if LABEL_SCHEME == "extended":
        from functional_groups import label_fragment
        return label_fragment(mol, group) or ["other"]
    atoms = set(group)
    labels = [name for name, patt in _FG_PATTERNS.items()
              if any(set(m) <= atoms for m in mol.GetSubstructMatches(patt))]
    drop = set().union(*(SUPPRESS.get(l, set()) for l in labels))
    labels = [l for l in labels if l not in drop]
    if labels:
        return labels
    if len(group) == 1:   # single-atom linker left between two BRICS cuts
        sym = mol.GetAtomWithIdx(group[0]).GetSymbol()
        return [f"{ELEMENT_NAMES.get(sym, sym)} linker atom"]
    if any(mol.GetAtomWithIdx(a).GetIsAromatic() for a in group):
        return ["aromatic ring"]
    if any(mol.GetAtomWithIdx(a).IsInRing() for a in group):
        return ["aliphatic ring"]
    if all(mol.GetAtomWithIdx(a).GetSymbol() == "C" for a in group):
        return ["alkyl chain"]
    return ["other"]


def fragment_features(mol, group, logp_contribs, mr_contribs, tpsa_contribs):
    """Morgan fingerprint of the fragment (centered on its atoms) + summed atom-level
    descriptor contributions computed on the parent molecule."""
    fp = _MORGAN.GetFingerprintAsNumPy(mol, fromAtoms=list(group)).astype(np.float32)
    atoms = [mol.GetAtomWithIdx(a) for a in group]
    desc = np.array([
        len(group),                                              # heavy atoms
        sum(a.GetIsAromatic() for a in atoms) / len(group),      # aromatic fraction
        sum(a.IsInRing() for a in atoms) / len(group),           # ring-atom fraction
        sum(a.GetSymbol() == "N" for a in atoms),
        sum(a.GetSymbol() == "O" for a in atoms),
        sum(a.GetSymbol() == "S" for a in atoms),
        sum(a.GetSymbol() in ("F", "Cl", "Br", "I") for a in atoms),
        sum(a.GetTotalNumHs() for a in atoms),
        sum(logp_contribs[a] for a in group),                    # Crippen logP contribution
        sum(mr_contribs[a] for a in group),                      # Crippen molar refractivity
        sum(tpsa_contribs[a] for a in group),                    # polar surface area contribution
    ], dtype=np.float32)
    return np.concatenate([fp, desc])


def bond_features(bond, ionic_mode=False):
    f = [float(bond.GetBondType() == t) for t in BOND_TYPES] + [float(bond.GetIsConjugated())]
    return f + [0.0] if ionic_mode else f


IONIC_EDGE = [0.0, 0.0, 0.0, 0.0, 0.0, 1.0]


def main_component(mol, components):
    """The parent molecule: the largest component that is not a known counter-ion or salt
    partner (so e.g. tosylic acid never wins over the drug it is paired with)."""
    def known(c):
        return Chem.MolFragmentToSmiles(mol, atomsToUse=list(c), canonical=True) in ION_NAMES
    cands = [i for i, c in enumerate(components) if not known(c)] or list(range(len(components)))
    return max(cands, key=lambda i: (len(components[i]),
                                     sum(mol.GetAtomWithIdx(a).GetSymbol() == "C" for a in components[i]), -i))


def ion_name(mol, comp_atoms):
    smi = Chem.MolFragmentToSmiles(mol, atomsToUse=list(comp_atoms), canonical=True)
    if smi in ION_NAMES:
        return ION_NAMES[smi]
    charge = sum(mol.GetAtomWithIdx(a).GetFormalCharge() for a in comp_atoms)
    return "cation" if charge > 0 else "anion" if charge < 0 else "salt partner / co-crystal component"


def ionic_links(mol, components, main, frag_of_atom):
    """Rules for linking each non-main component (counter-ion, salt partner, solvent) to the
    rest of the molecule, since no covalent bond connects them:
      1. charged component -> fragments holding atoms of opposite formal charge
         (prefer the main component; otherwise any other component)
      2. neutral component (e.g. 'Cl' for an HCl salt, an acid partner, water) ->
         the main-component fragment holding the most basic nitrogen (amine first,
         then aromatic N); if there is none, the main component's largest fragment
    Returns a set of (fragment_u, fragment_v) pairs."""
    links = set()
    main_atoms = set(components[main])

    def frag_sizes():
        sizes = {}
        for a in main_atoms:
            sizes[frag_of_atom[a]] = sizes.get(frag_of_atom[a], 0) + 1
        return sizes

    for ci, comp in enumerate(components):
        if ci == main:
            continue
        src = frag_of_atom[comp[0]]            # non-main components are single nodes
        charge = sum(mol.GetAtomWithIdx(a).GetFormalCharge() for a in comp)
        targets = []
        if charge != 0:
            sign = 1 if charge > 0 else -1
            opposite = [a.GetIdx() for a in mol.GetAtoms()
                        if a.GetFormalCharge() * sign < 0 and a.GetIdx() not in comp]
            in_main = [a for a in opposite if a in main_atoms]
            targets = sorted({frag_of_atom[a] for a in (in_main or opposite)})
        if not targets:
            for patt in (_BASIC_N, _AROM_N):
                hits = [m[0] for m in mol.GetSubstructMatches(patt) if m[0] in main_atoms]
                if hits:
                    targets = [frag_of_atom[hits[0]]]
                    break
        if not targets:
            sizes = frag_sizes()
            targets = [max(sizes, key=sizes.get)]
        for t in targets:
            if t != src:
                links.add((src, t))
    return links


def fragment_graph(smiles, ionic=False):
    """Everything needed to build one fragment graph, as plain Python/numpy.
    Returns None for invalid SMILES. See the module docstring for the two modes."""
    mol, changed = prepare_mol(smiles, ionic)
    if mol is None:
        return None

    components = [list(c) for c in Chem.GetMolFrags(mol)]
    main = main_component(mol, components)
    allowed = set(components[main]) if ionic else None
    groups, frag_of_atom, broken = brics_partition(mol, allowed)
    comp_of_atom = {a: ci for ci, c in enumerate(components) for a in c}

    crippen = rdMolDescriptors._CalcCrippenContribs(mol)
    logp = [c[0] for c in crippen]
    mr = [c[1] for c in crippen]
    tpsa = list(rdMolDescriptors._CalcTPSAContribs(mol))

    texts, labels, counterion = [], [], []
    for group in groups:
        frag_smi = Chem.MolFragmentToSmiles(mol, atomsToUse=list(group), canonical=True)
        is_ci = ionic and comp_of_atom[group[0]] != main
        if is_ci:
            fg = [ion_name(mol, group)]
            texts.append(f"{frag_smi} (counter-ion: {fg[0]})")
        else:
            fg = functional_groups(mol, group)
            texts.append(f"{frag_smi} ({', '.join(fg)})")
        labels.append(fg)
        counterion.append(is_ci)

    x = np.stack([fragment_features(mol, g, logp, mr, tpsa) for g in groups])
    if ionic:   # net formal charge of the fragment, and whether it is a counter-ion node
        extra = np.array([[sum(mol.GetAtomWithIdx(a).GetFormalCharge() for a in g), float(ci)]
                          for g, ci in zip(groups, counterion)], dtype=np.float32)
        x = np.concatenate([x, extra], axis=1)

    edges, edge_attr = [], []
    for i, j in broken:
        u, v = frag_of_atom[i], frag_of_atom[j]
        feats = bond_features(mol.GetBondBetweenAtoms(i, j), ionic)
        edges += [(u, v), (v, u)]
        edge_attr += [feats, feats]
    n_ionic = 0
    if ionic and len(components) > 1:
        for u, v in sorted(ionic_links(mol, components, main, frag_of_atom)):
            edges += [(u, v), (v, u)]
            edge_attr += [IONIC_EDGE, IONIC_EDGE]
            n_ionic += 1

    return {
        "smiles": Chem.MolToSmiles(mol),
        "ionic": ionic,
        "salt_stripped": changed and not ionic,
        "metal_disconnected": changed and ionic,
        "n_components": len(components),
        "n_counterion_nodes": sum(counterion),
        "n_ionic_edges": n_ionic,
        "num_fragments": len(groups),
        "fragment_text": texts,
        "fragment_labels": labels,
        "counterion": counterion,
        "atom_groups": [list(g) for g in groups],
        "frag_of_atom": frag_of_atom,
        "x": x,
        "edges": edges,
        "edge_attr": edge_attr,
    }


# ----------------------------------------------------------------------------- data

CSV_COLUMNS = {
    # name: (SMILES column, label column) in the standard MoleculeNet CSVs
    "BBBP": ("smiles", "p_np"),
    "BACE": ("mol", "Class"),
    "ESOL": ("smiles", "measured log solubility in mols per litre"),
}


def scaffold_split(smiles_list, frac_train=0.8, frac_valid=0.1):
    """Deterministic Bemis-Murcko scaffold split (same procedure as DeepChem's
    ScaffoldSplitter): group by scaffold, largest groups go to train first."""
    from rdkit.Chem.Scaffolds import MurckoScaffold
    groups = {}
    for i, smi in enumerate(smiles_list):
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        scaf = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
        groups.setdefault(scaf, []).append(i)
    ordered = sorted(groups.values(), key=lambda g: (len(g), g[0]), reverse=True)
    n = sum(len(g) for g in ordered)
    train, valid, test = [], [], []
    for g in ordered:
        if len(train) + len(g) <= frac_train * n:
            train += g
        elif len(valid) + len(g) <= frac_valid * n:
            valid += g
        else:
            test += g
    return train, valid, test


def load_split_csv(name, csv_path):
    """Load a MoleculeNet CSV you already have and scaffold-split it locally."""
    smi_col, y_col = CSV_COLUMNS[name]
    df = pd.read_csv(csv_path)
    smiles, ys = df[smi_col].astype(str).tolist(), df[y_col].tolist()
    idx = scaffold_split(smiles)
    splits = {s: [(smiles[i], ys[i]) for i in ids]
              for s, ids in zip(("train", "valid", "test"), idx)}
    return splits, DATASETS[name][1]


CSV_PATHS = {}   # filled from --csv arguments


def load_split(name):
    """MoleculeNet via DeepChem with a scaffold split, or a local CSV if given."""
    if name in CSV_PATHS:
        return load_split_csv(name, CSV_PATHS[name])
    import deepchem as dc
    loader_name, task_type = DATASETS[name]
    tasks, (train, valid, test), _ = getattr(dc.molnet, loader_name)(
        featurizer="Raw", splitter="scaffold", transformers=[])
    splits = {}
    for split_name, ds in (("train", train), ("valid", valid), ("test", test)):
        y = ds.y[:, 0] if ds.y.ndim > 1 else ds.y      # BBBP, BACE, ESOL are single-task
        splits[split_name] = list(zip(ds.ids, y))
    return splits, task_type


def to_pyg(g, y, task_type):
    import torch
    from torch_geometric.data import Data
    edge_index = (torch.tensor(g["edges"], dtype=torch.long).t().contiguous()
                  if g["edges"] else torch.zeros((2, 0), dtype=torch.long))
    edge_dim = len(BOND_TYPES) + 1 + int(g["ionic"])
    edge_attr = (torch.tensor(g["edge_attr"], dtype=torch.float)
                 if g["edge_attr"] else torch.zeros((0, edge_dim)))
    return Data(
        x=torch.tensor(g["x"], dtype=torch.float),
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.tensor([float(y)], dtype=torch.float),
        smiles=g["smiles"],
        fragment_text=g["fragment_text"],
        atom_groups=g["atom_groups"],
        counterion=torch.tensor(g["counterion"], dtype=torch.bool),
        ionic=g["ionic"],
        label_version=label_version(),
        num_nodes=g["num_fragments"],
    )


# ----------------------------------------------------------------------------- main

def mode_paths(ionic):
    """Output locations for each mode, so the two variants never overwrite each other."""
    return ("frag_graphs_ions", "fragment_stats_ions.csv") if ionic else ("frag_graphs", "fragment_stats.csv")


def run(names, build, out_dir=None, ionic=False):
    default_dir, stats_path = mode_paths(ionic)
    out_dir = out_dir or default_dir
    rows = []
    for name in names:
        splits, task_type = load_split(name)
        counts, stripped, invalid, multi, ci_nodes, metal = [], 0, 0, 0, 0, 0
        for split_name, items in splits.items():
            graphs = []
            for smiles, y in items:
                g = fragment_graph(smiles, ionic)
                if g is None:
                    invalid += 1
                    continue
                counts.append(g["num_fragments"])
                stripped += g["salt_stripped"]
                multi += g["n_components"] > 1
                ci_nodes += g["n_counterion_nodes"]
                metal += g["metal_disconnected"]
                if build:
                    graphs.append(to_pyg(g, y, task_type))
            if build:
                import torch
                os.makedirs(out_dir, exist_ok=True)
                torch.save(graphs, os.path.join(out_dir, f"{name}_{split_name}.pt"))
                with open(os.path.join(out_dir, f"{name}_LABELS.txt"), "w") as f:
                    f.write(label_version())
        c = np.array(counts)
        row = {
            "dataset": name,
            "mode": "BRICS + ionic" if ionic else "BRICS (salts stripped)",
            "task": task_type,
            "molecules": len(c),
            "train/valid/test": "/".join(str(len(v)) for v in splits.values()),
            "mean_fragments": c.mean(),
            "std_fragments": c.std(),
            "median_fragments": float(np.median(c)),
            "max_fragments": int(c.max()),
            "pct_le_12": 100 * (c <= MAX_FRAGMENTS_FOR_EXACT_SHAPLEY).mean(),
            "pct_single_fragment": 100 * (c == 1).mean(),
            "salts_stripped": stripped,
            "invalid_smiles": invalid,
        }
        if ionic:
            row.update(pct_with_counterions=100 * multi / len(c), counterion_nodes=ci_nodes,
                       metal_bonds_split=metal)
        rows.append(row)
        print(f"done: {name}")

    df = pd.DataFrame(rows)
    if os.path.exists(stats_path):    # keep rows for datasets not rebuilt this time
        old = pd.read_csv(stats_path)
        df = pd.concat([old[~old["dataset"].isin(df["dataset"])], df], ignore_index=True)
    df.to_csv(stats_path, index=False)
    pd.set_option("display.width", 200)
    print("\n", df.round(2).to_string(index=False))
    print("\nLaTeX rows (Dataset & Task & #Mol & Frags mean±std & Max & % ≤12 & % single):")
    for r in rows:
        print(f"{r['dataset']} & {r['task']} & {r['molecules']} & "
              f"{r['mean_fragments']:.1f} $\\pm$ {r['std_fragments']:.1f} & {r['max_fragments']} & "
              f"{r['pct_le_12']:.1f} & {r['pct_single_fragment']:.1f} \\\\")


def demo(smiles, ionic=False):
    g = fragment_graph(smiles, ionic)
    if g is None:
        print("Invalid SMILES")
        return
    note = ("  (counter-ion removed)" if g["salt_stripped"] else
            f"  ({g['n_components']} components, {g['n_ionic_edges']} ionic edges)" if ionic else "")
    print(f"{g['smiles']}  ->  {g['num_fragments']} fragments{note}")
    for i, t in enumerate(g["fragment_text"]):
        print(f"  node {i}: {t}")
    print("  edges:", sorted({tuple(sorted(e)) + (("ionic",) if a[-1] == 1.0 and ionic else ())
                              for e, a in zip(g["edges"], g["edge_attr"])}))
    print("  node feature size:", g["x"].shape[1], "| edge feature size:", len(g["edge_attr"][0]) if g["edge_attr"] else "-")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--demo", type=str, help="print the fragment graph for one SMILES")
    ap.add_argument("--stats", action="store_true", help="fragment statistics for the datasets")
    ap.add_argument("--build", action="store_true", help="also save PyG graphs (needs torch_geometric)")
    ap.add_argument("--ionic", action="store_true",
                    help="BRICS + ionic decomposition (keep counter-ions as nodes); default strips them")
    ap.add_argument("--datasets", nargs="*", default=list(DATASETS))
    ap.add_argument("--out", default=None, help="graph folder (default frag_graphs or frag_graphs_ions)")
    ap.add_argument("--csv", nargs="*", default=[], metavar="NAME=PATH",
                    help="use local CSVs instead of DeepChem, e.g. BBBP=BBBP.csv BACE=bace.csv ESOL=delaney-processed.csv")
    args = ap.parse_args()
    for item in args.csv:
        k, v = item.split("=", 1)
        CSV_PATHS[k.upper()] = v
    if args.demo:
        demo(args.demo, args.ionic)
    elif args.stats or args.build:
        run(args.datasets, args.build, args.out, args.ionic)
    else:
        ap.print_help()

In [ ]:
%%writefile functional_groups.py
"""Functional-group and ring labels for fragments, plus list-free detection.

Three sources, combined in label_fragment():

1. Curated SMARTS (rule list). RDKit's FragmentDescriptors.csv (85 groups; the basis of
   RDKit's fr_* descriptors), minus entries that are metabolic-site heuristics or counts rather
   than functional groups, plus groups that list lacks (carbamate, sulfonic acid, sulfoxide,
   acyl halide, anhydride, alkene, alkyne, CF3, ...).
2. Ring names. Each ring of the fragment is matched to a table of common aromatic and
   aliphatic rings (pyridine, piperazine, ...); unknown rings are described by size/aromaticity.
3. Ertl's algorithm (J. Cheminform. 2017, 9:36) - no predefined list. Atoms are marked by
   general rules (heteroatoms; carbons in C=C, C#C or double/triple bonds to heteroatoms;
   acetal-type sp3 carbons; atoms of 3-membered heterocycles); connected marked atoms form one
   functional group, reported as a generalized SMILES type. Any group the curated list misses
   still gets a label this way ("group N=C(N)S" etc.).

Redundant labels are removed by subsumption: a label is dropped when every one of its matches
lies inside a strictly larger match of another label (so "carboxylic acid" absorbs "hydroxyl"
and "carbonyl" without hand-written exceptions).

mine_groups(smiles) counts Ertl types across a dataset: a data-driven vocabulary of the groups
that actually occur, with how often the curated list already names them.
"""

import csv
import os
from collections import Counter

from rdkit import Chem, RDConfig, RDLogger

RDLogger.DisableLog("rdApp.*")

LABEL_VERSION = "v2"   # bump when labels change; graphs/projectors/explanations are tagged with it

# ----------------------------------------------------------------------------- curated SMARTS

# RDKit entries that are not functional groups (metabolism sites, bicyclic flag, duplicates,
# overly broad counts) are skipped.
_SKIP = {"fr_C=O_noCOO", "fr_COO2", "fr_NH0", "fr_NH1", "fr_NH2", "fr_Ar_N", "fr_Ar_NH", "fr_Nhpyrrole",
         "fr_bicyclic", "fr_para_hydroxylation", "fr_allylic_oxid", "fr_aryl_methyl",
         "fr_Ndealkylation1", "fr_Ndealkylation2", "fr_HOCCN", "fr_ketone_Topliss",
         "fr_phenol_noOrthoHbond", "fr_Al_OH_noTert", "fr_nitro_arom_nonortho", "fr_ArN",
         "fr_unbrch_alkane", "fr_benzodiazepine", "fr_COO", "fr_C=O", "fr_halogen",
         "fr_imidazole", "fr_furan", "fr_thiophene", "fr_thiazole", "fr_oxazole", "fr_pyridine",
         "fr_piperdine", "fr_piperzine", "fr_morpholine", "fr_tetrazole", "fr_benzene"}   # rings: see RING_NAMES
# Shorter display names for RDKit entries
_RENAME = {"fr_Al_OH": "aliphatic hydroxyl", "fr_Ar_OH": "aromatic hydroxyl", "fr_Al_COO": "aliphatic carboxylic acid",
           "fr_Ar_COO": "aromatic carboxylic acid", "fr_quatN": "quaternary nitrogen", "fr_Imine": "imine",
           "fr_hdrzine": "hydrazine", "fr_hdrzone": "hydrazone", "fr_N-O": "hydroxylamine",
           "fr_priamide": "primary amide", "fr_guanido": "guanidine", "fr_isocyan": "isocyanate",
           "fr_isothiocyan": "isothiocyanate", "fr_sulfide": "thioether", "fr_SH": "thiol",
           "fr_C=S": "thiocarbonyl", "fr_sulfonamd": "sulfonamide", "fr_prisulfonamd": "primary sulfonamide",
           "fr_barbitur": "barbiturate", "fr_term_acetylene": "terminal alkyne", "fr_lactam": "beta-lactam",
           "fr_lactone": "lactone", "fr_epoxide": "epoxide", "fr_phos_acid": "phosphoric acid",
           "fr_phos_ester": "phosphate ester", "fr_nitro_arom": "nitroarene", "fr_dihydropyridine": "dihydropyridine",
           "fr_alkyl_carbamate": "alkyl carbamate", "fr_alkyl_halide": "alkyl halide", "fr_methoxy": "methoxy",
           "fr_aniline": "aniline (aryl amine)", "fr_ether": "ether", "fr_nitroso": "nitroso",
           "fr_amidine": "amidine", "fr_azide": "azide", "fr_azo": "azo", "fr_diazo": "diazo", "fr_imide": "imide",
           "fr_oxime": "oxime", "fr_sulfone": "sulfone", "fr_urea": "urea", "fr_nitro": "nitro"}
# Groups the RDKit list does not have
EXTRA_SMARTS = {
    "carboxylic acid": "[CX3](=O)[OX2H1]",
    "carboxylate": "[CX3](=O)[O-]",
    "carbamate": "[NX3][CX3](=O)[OX2][#6]",
    "carbonate": "[OX2][CX3](=O)[OX2]",
    "acyl halide": "[CX3](=O)[F,Cl,Br,I]",
    "anhydride": "[CX3](=O)[OX2][CX3]=O",
    "sulfonic acid": "[SX4](=O)(=O)[OX2H1,OX1-]",
    "sulfonate ester": "[SX4](=O)(=O)[OX2][#6]",
    "sulfoxide": "[$([SX3](=O)([#6])[#6]),$([SX3+]([O-])([#6])[#6])]",
    "sulfonyl": "[SX4](=O)(=O)",
    "disulfide": "[SX2][SX2]",
    "primary amine": "[NX3;H2;!$(N-C=[O,S,N]);!$(N-a);!$(N-S)][CX4]",
    "secondary amine": "[NX3;H1;!$(N-C=[O,S,N]);!$(N-a);!$(N-S)]([CX4])[CX4]",
    "tertiary amine": "[NX3;H0;!$(N-C=[O,S,N]);!$(N-a);!$(N-S);!$(N=*);!$(N#*)]([CX4])([CX4])[CX4]",
    "protonated amine (ammonium)": "[NX4+;!$(N~O)]",
    "enamine": "[NX3][CX3]=[CX3]",
    "enol ether": "[OX2]([#6])[CX3]=[CX3]",
    "alkene": "[CX3]=[CX3]",
    "alkyne": "[CX2]#[CX2]",
    "trifluoromethyl": "C(F)(F)F",
    "aryl halide": "c[F,Cl,Br,I]",
    "acetal / ketal": "[CX4]([OX2][#6])[OX2][#6]",
    "hemiacetal": "[CX4]([OX2H])[OX2][#6]",
    "boronic acid": "[BX3]([OX2H])[OX2H]",
    "N-oxide": "[#7+;!$([#7+](=O)[O-]);!$([#7+]=O)][OX1-]",
    "ring carbonyl (aromatic C=O)": "[#6;a]=[OX1]",
}


def _load_rdkit_groups():
    path = os.path.join(RDConfig.RDDataDir, "FragmentDescriptors.csv")
    out = {}
    if not os.path.exists(path):
        return out
    with open(path) as f:
        for row in csv.reader((l for l in f if l.strip() and not l.startswith("#")), delimiter="\t"):
            key, desc, smarts = row[0], row[1], row[2]
            if key in _SKIP:
                continue
            name = _RENAME.get(key) or desc.replace("Number of ", "").rstrip("s").strip().lower()
            out[name] = smarts
    return out


CURATED = {**_load_rdkit_groups(), **EXTRA_SMARTS}
_PATTERNS = {}
for _name, _s in CURATED.items():
    _p = Chem.MolFromSmarts(_s)
    if _p is not None:
        _PATTERNS[_name] = _p

# ----------------------------------------------------------------------------- rings

RING_NAMES = {
    # aromatic 6
    "c1ccccc1": "benzene ring", "c1ccncc1": "pyridine ring", "c1cncnc1": "pyrimidine ring",
    "c1cnccn1": "pyrazine ring", "c1ccnnc1": "pyridazine ring", "c1ncncn1": "triazine ring",
    "O=c1cccc[nH]1": "pyridone ring",
    # aromatic 5
    "c1cc[nH]c1": "pyrrole ring", "c1ccoc1": "furan ring", "c1ccsc1": "thiophene ring",
    "c1c[nH]cn1": "imidazole ring", "c1cn[nH]c1": "pyrazole ring", "c1cocn1": "oxazole ring",
    "c1cnoc1": "isoxazole ring", "c1cscn1": "thiazole ring", "c1cnsc1": "isothiazole ring",
    "c1nc[nH]n1": "1,2,4-triazole ring", "c1cn[nH]n1": "1,2,3-triazole ring", "c1nn[nH]n1": "tetrazole ring",
    "c1nnco1": "1,3,4-oxadiazole ring", "c1ncon1": "1,2,4-oxadiazole ring", "c1nncs1": "thiadiazole ring",
    # aliphatic
    "C1CC1": "cyclopropane", "C1CCC1": "cyclobutane", "C1CCCC1": "cyclopentane", "C1CCCCC1": "cyclohexane",
    "C1CCCCCC1": "cycloheptane", "C1CCNC1": "pyrrolidine ring", "C1CCNCC1": "piperidine ring",
    "C1CNCCN1": "piperazine ring", "C1COCCN1": "morpholine ring", "C1CCOC1": "tetrahydrofuran ring",
    "C1CCOCC1": "tetrahydropyran ring", "C1CSCN1": "thiazolidine ring", "C1CO1": "oxirane ring",
    "C1CN1": "aziridine ring", "C1COCO1": "dioxolane ring", "C1CCSC1": "tetrahydrothiophene ring",
    "C1=CCCCC1": "cyclohexene", "C1CNC1": "azetidine ring", "C1=CCCC1": "cyclopentene",
    "C1=CNC=CC1": "1,4-dihydropyridine ring", "C1=CCNCC1": "tetrahydropyridine ring",
    "O=C1CCCN1": "pyrrolidinone (lactam) ring", "C1CCNCC1": "piperidine ring",
}


def _canon(smi):
    m = Chem.MolFromSmiles(smi)
    return Chem.MolToSmiles(m) if m is not None else smi


RING_NAMES = {_canon(k): v for k, v in RING_NAMES.items()}   # keys in RDKit canonical form


def _ring_key(mol, ring):
    """Ring atoms only, all exocyclic substituents removed, charges and H counts reset."""
    em = Chem.RWMol()
    idx = {}
    for a in ring:
        at = mol.GetAtomWithIdx(a)
        n = Chem.Atom(at.GetAtomicNum())
        n.SetIsAromatic(at.GetIsAromatic())
        idx[a] = em.AddAtom(n)
    for i, a in enumerate(ring):
        b = ring[(i + 1) % len(ring)]
        bond = mol.GetBondBetweenAtoms(a, b)
        em.AddBond(idx[a], idx[b], bond.GetBondType())
    m = em.GetMol()
    try:
        m.UpdatePropertyCache(strict=False)
        Chem.SanitizeMol(m, Chem.SanitizeFlags.SANITIZE_ALL ^ Chem.SanitizeFlags.SANITIZE_PROPERTIES, catchErrors=True)
        smi = Chem.MolToSmiles(m)
        # aromatic N with implicit H: try the [nH] form so pyrrole/imidazole-type rings match the table
        if smi not in RING_NAMES:
            for alt in (smi.replace("n", "[nH]", 1),):
                mm = Chem.MolFromSmiles(alt)
                if mm is not None and Chem.MolToSmiles(mm) in RING_NAMES:
                    return Chem.MolToSmiles(mm)
        return smi
    except Exception:
        return Chem.MolFragmentToSmiles(mol, atomsToUse=list(ring))


def ring_labels(mol, atoms):
    atoms = set(atoms)
    ri = mol.GetRingInfo()
    rings = [r for r in ri.AtomRings() if set(r) <= atoms]
    labels = []
    for r in rings:
        key = _ring_key(mol, r)
        if key in RING_NAMES:
            labels.append(RING_NAMES[key])
        else:
            arom = all(mol.GetAtomWithIdx(a).GetIsAromatic() for a in r)
            het = sorted({mol.GetAtomWithIdx(a).GetSymbol() for a in r} - {"C"})
            labels.append(f"{len(r)}-membered {'aromatic' if arom else 'aliphatic'} "
                          + (f"{'/'.join(het)}-heterocycle" if het else "carbocycle"))
    fused = any(len(set(r1) & set(r2)) >= 2 for i, r1 in enumerate(rings) for r2 in rings[i + 1:])
    if fused:
        labels.append("fused ring system")
    return labels


# ----------------------------------------------------------------------------- Ertl IFG

def ertl_groups(mol):
    """Ertl's functional-group identification. Returns a list of (atom set, type SMILES)."""
    marked = set()
    for at in mol.GetAtoms():
        if at.GetAtomicNum() not in (1, 6):                       # heteroatoms
            marked.add(at.GetIdx())
    for b in mol.GetBonds():                                      # C=C, C#C, C=X, C#X (non-aromatic)
        if b.GetBondType() in (Chem.BondType.DOUBLE, Chem.BondType.TRIPLE) and not b.GetIsAromatic():
            marked.update((b.GetBeginAtomIdx(), b.GetEndAtomIdx()))
    for at in mol.GetAtoms():                                     # acetal-type sp3 carbons
        if at.GetAtomicNum() == 6 and at.GetHybridization() == Chem.HybridizationType.SP3:
            het = [n for n in at.GetNeighbors() if n.GetAtomicNum() in (7, 8, 16)
                   and all(bb.GetBondType() == Chem.BondType.SINGLE for bb in n.GetBonds())]
            if len(het) >= 2:
                marked.add(at.GetIdx())
    for r in mol.GetRingInfo().AtomRings():                       # oxirane / aziridine / thiirane
        if len(r) == 3 and any(mol.GetAtomWithIdx(a).GetAtomicNum() in (7, 8, 16) for a in r):
            marked.update(r)
    # connected components of marked atoms
    groups, seen = [], set()
    for a in marked:
        if a in seen:
            continue
        stack, comp = [a], set()
        while stack:
            x = stack.pop()
            if x in comp:
                continue
            comp.add(x)
            stack += [n.GetIdx() for n in mol.GetAtomWithIdx(x).GetNeighbors() if n.GetIdx() in marked]
        seen |= comp
        groups.append(comp)
    out = []
    for comp in groups:
        try:
            t = Chem.MolFragmentToSmiles(mol, atomsToUse=sorted(comp), canonical=True)
        except Exception:
            t = "?"
        out.append((comp, t))
    return out


# ----------------------------------------------------------------------------- labelling

def _subsume(matches):
    """matches: {label: [atom sets]}. Drop a label when every one of its matches lies inside a
    larger match of another label, or coincides with a match of a more specific label
    (longer SMARTS), e.g. 'sulfonamide' gives way to 'primary sulfonamide'."""
    spec = {l: len(CURATED.get(l, "")) for l in matches}
    keep = []
    for lab, ms in matches.items():
        if "$(" in CURATED.get(lab, ""):      # recursive SMARTS match only an anchor atom: keep
            keep.append(lab)
            continue
        def absorbed(m):
            for l2, ms2 in matches.items():
                if l2 == lab:
                    continue
                for o in ms2:
                    if m < o or (m == o and spec[l2] > spec[lab]):
                        return True
            return False
        if all(absorbed(m) for m in ms):
            continue
        keep.append(lab)
    return keep


ELEMENT_NAMES = {"N": "nitrogen", "O": "oxygen", "S": "sulfur", "C": "carbon", "P": "phosphorus"}


def label_fragment(mol, atoms, max_labels=6):
    """Readable labels for the fragment made of `atoms` (indices into mol):
      - single-atom fragments: '<element> linker atom'
      - rings in the fragment (named when known)
      - curated groups fully inside the fragment
      - 'part of <group>' for curated groups that BRICS cut through (e.g. an amide split
        between its C=O and N fragments)
      - Ertl groups (list-free) for anything the curated list did not cover"""
    atoms = set(atoms)
    inside, crossing = {}, {}
    for name, patt in _PATTERNS.items():
        for m in mol.GetSubstructMatches(patt):
            m = set(m)
            if m <= atoms:
                inside.setdefault(name, []).append(m)
            elif m & atoms:
                crossing.setdefault(name, []).append(m)
    fg_labels = _subsume(inside)
    marked = set().union(*(c for c, _ in ertl_groups(mol))) if mol.GetNumAtoms() else set()
    inside_sets = [m for l in fg_labels for m in inside[l]]
    # 'part of X' only when the fragment holds part of X's functional core (a heteroatom or a
    # multiply bonded carbon), not just a carbon it is attached to, and nothing inside already covers it
    crossing = {l: [m for m in ms if (m & atoms & marked) and not any((m & atoms) <= im for im in inside_sets)]
                for l, ms in crossing.items()}
    crossing = {l: ms for l, ms in crossing.items() if ms}
    part_labels = [f"part of {l}" for l in _subsume(crossing) if l not in fg_labels]
    named_atoms = set()
    for l in fg_labels:
        for m in inside[l]:
            named_atoms |= m
    for l in crossing:
        for m in crossing[l]:
            named_atoms |= (m & atoms)
    ring_l = ring_labels(mol, atoms)
    ring_atoms = {a for r in mol.GetRingInfo().AtomRings() if set(r) <= atoms for a in r}
    extra = []
    for comp, t in ertl_groups(mol):
        local = comp & atoms
        if not local or local & named_atoms:
            continue
        if local <= ring_atoms and all(mol.GetAtomWithIdx(a).GetIsAromatic() for a in local):
            continue      # aromatic ring heteroatoms: already described by the ring name
        if len(local) == 1 and mol.GetAtomWithIdx(next(iter(local))).GetSymbol() in ("F", "Cl", "Br", "I"):
            extra.append("halogen")
            continue
        if comp <= atoms:
            extra.append(f"group {t}")
        else:
            part = Chem.MolFragmentToSmiles(mol, atomsToUse=sorted(local), canonical=True)
            extra.append(f"part of group {t}" if len(t) <= 12 else f"group {part} (cut)")
    head = []
    if len(atoms) == 1:
        sym = mol.GetAtomWithIdx(next(iter(atoms))).GetSymbol()
        head = [f"{ELEMENT_NAMES.get(sym, sym)} linker atom"]
    fg_labels = [l for l in fg_labels if not any(l in r for r in ring_l)]   # e.g. 'dihydropyridine'
    labels = list(dict.fromkeys(head + ring_l + fg_labels + part_labels + extra))
    if not labels and all(mol.GetAtomWithIdx(a).GetSymbol() == "C" for a in atoms):
        labels = ["alkyl chain"]
    return labels[:max_labels]


def mine_groups(smiles_list, top=40):
    """Data-driven vocabulary: Ertl group types found in a set of molecules, how many molecules
    contain each, and whether the curated list names them. Returns a pandas DataFrame."""
    import pandas as pd
    counts, named = Counter(), Counter()
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        seen = set()
        cover = set(a for p in _PATTERNS.values() for m in mol.GetSubstructMatches(p) for a in m)
        named_rings = set(a for r in mol.GetRingInfo().AtomRings() if _ring_key(mol, r) in RING_NAMES for a in r)
        for comp, t in ertl_groups(mol):
            if t in seen:
                continue
            seen.add(t)
            counts[t] += 1
            # same rule as label_fragment: overlapping a curated match, or sitting in a named ring
            if comp & cover or comp <= named_rings:
                named[t] += 1
    rows = [{"ertl_type": t, "molecules": c, "pct_named_by_curated": 100 * named[t] / c}
            for t, c in counts.most_common(top)]
    return pd.DataFrame(rows)

## 3. Check CSV columns

Finds the SMILES and label column in each file. If a label column can't be found, the cell stops and prints the header so you can set it by hand.

In [ ]:
import importlib, fragment_graphs as fg
importlib.reload(fg)

LABEL_CANDIDATES = {
    "BBBP": ["p_np", "label", "y"],
    "BACE": ["Class", "class", "label", "y"],
    "ESOL": ["measured log solubility in mols per litre", "logSolubility", "measured_log_solubility",
             "logS", "solubility", "y"],
}
for name, path in CSVS.items():
    cols = list(pd.read_csv(path, nrows=1).columns)
    smi = next((c for c in cols if c.lower() in ("smiles", "mol")), None)
    lab = next((c for c in LABEL_CANDIDATES[name] if c in cols), None)
    if smi is None or lab is None:
        raise KeyError(f"{name}: couldn't find columns. Header is {cols}. "
                       f"Set fg.CSV_COLUMNS['{name}'] = (smiles_col, label_col) and rerun.")
    fg.CSV_COLUMNS[name] = (smi, lab)
    fg.CSV_PATHS[name] = path
    print(f"{name:5s} smiles='{smi}'  label='{lab}'")

## 4. Build fragment graphs + Table 1 statistics

In [ ]:
fg.run(list(CSVS), build=True, out_dir="frag_graphs")
stats = pd.read_csv("fragment_stats.csv")

In [ ]:
# Look at one molecule's graph to sanity-check the node text
g = torch.load("frag_graphs/BBBP_test.pt", weights_only=False)[0]
print(g.smiles)
for i, t in enumerate(g.fragment_text): print(f"  node {i}: {t}")
print(g)

## 5. Figure: fragments per molecule

In [ ]:
import matplotlib.pyplot as plt
os.makedirs("figures", exist_ok=True)
fig, ax = plt.subplots(figsize=(4.2, 2.8))
for name in CSVS:
    counts = [d.num_nodes for s in ("train", "valid", "test")
              for d in torch.load(f"frag_graphs/{name}_{s}.pt", weights_only=False)]
    ax.hist(counts, bins=range(1, max(counts) + 2), histtype="step", linewidth=1.5, label=name, density=True)
ax.axvline(12.5, color="gray", linestyle="--", linewidth=1)
ax.text(12.8, ax.get_ylim()[1] * 0.85, "exact Shapley\nlimit (12)", fontsize=7, color="gray")
ax.set_xlabel("BRICS fragments per molecule"); ax.set_ylabel("fraction of molecules")
ax.legend(frameon=False, fontsize=8); fig.tight_layout()
fig.savefig("figures/fragment_hist.pdf"); plt.show()

## 6. Base GIN on fragment graphs

Fragment features → MLP compression → 3 GINE layers (edge features = broken-bond type) → attention readout → prediction head. ESOL labels are standardized for training and RMSE is reported in original units.

In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GINEConv
from torch_geometric.nn.aggr import AttentionalAggregation
from sklearn.metrics import roc_auc_score

class FragGIN(nn.Module):
    def __init__(self, in_dim, edge_dim=5, hidden=128, layers=3, dropout=0.2):
        super().__init__()
        self.compress = nn.Sequential(nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
                                      nn.Linear(hidden, hidden))
        self.convs, self.norms = nn.ModuleList(), nn.ModuleList()
        for _ in range(layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=edge_dim))
            self.norms.append(nn.BatchNorm1d(hidden))
        self.readout = AttentionalAggregation(gate_nn=nn.Linear(hidden, 1))
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
        self.dropout = dropout

    def embed(self, data):
        """Fragment (node) embeddings and molecule (graph) embedding — what GSPELL's projector will use."""
        h = self.compress(data.x)
        for conv, norm in zip(self.convs, self.norms):
            h = h + F.dropout(F.relu(norm(conv(h, data.edge_index, data.edge_attr))), self.dropout, self.training)
        return h, self.readout(h, data.batch)

    def forward(self, data):
        return self.head(self.embed(data)[1]).squeeze(-1)


def load_graphs(name):
    return [torch.load(f"frag_graphs/{name}_{s}.pt", weights_only=False) for s in ("train", "valid", "test")]


@torch.no_grad()
def evaluate(model, loader, task, mu, sd):
    model.eval(); preds, ys = [], []
    for b in loader:
        b = b.to(DEVICE); out = model(b)
        preds.append(torch.sigmoid(out) if task == "classification" else out * sd + mu)
        ys.append(b.y)
    p, y = torch.cat(preds).cpu().numpy(), torch.cat(ys).cpu().numpy()
    return roc_auc_score(y, p) if task == "classification" else float(np.sqrt(((p - y) ** 2).mean()))


def train_one(name, task, seed, epochs=150, patience=25, lr=1e-3):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    train, valid, test = load_graphs(name)
    ys = torch.cat([d.y for d in train])
    mu, sd = (ys.mean().item(), ys.std().item()) if task == "regression" else (0.0, 1.0)
    tl = DataLoader(train, batch_size=64, shuffle=True, drop_last=True)
    vl, te = DataLoader(valid, batch_size=256), DataLoader(test, batch_size=256)

    model = FragGIN(in_dim=train[0].x.size(1)).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    better = (lambda a, b: a > b) if task == "classification" else (lambda a, b: a < b)
    best, best_state, wait = None, None, 0
    for epoch in range(epochs):
        model.train()
        for b in tl:
            b = b.to(DEVICE); opt.zero_grad()
            out = model(b)
            loss = (F.binary_cross_entropy_with_logits(out, b.y) if task == "classification"
                    else F.mse_loss(out, (b.y - mu) / sd))
            loss.backward(); opt.step()
        v = evaluate(model, vl, task, mu, sd)
        if best is None or better(v, best):
            best, wait = v, 0
            best_state = {k: t.detach().cpu().clone() for k, t in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    model.load_state_dict(best_state)
    test_metric = evaluate(model, te, task, mu, sd)
    os.makedirs("checkpoints", exist_ok=True)
    torch.save({"state_dict": best_state, "mu": mu, "sd": sd, "in_dim": train[0].x.size(1)},
               f"checkpoints/{name}_gin_seed{seed}.pt")
    return best, test_metric, epoch + 1

In [ ]:
SEEDS = [0, 1, 2]
rows = []
for name in CSVS:
    task = fg.DATASETS[name][1]
    metric = "ROC-AUC" if task == "classification" else "RMSE"
    tests = []
    for seed in SEEDS:
        val, test_m, ep = train_one(name, task, seed)
        tests.append(test_m)
        print(f"{name} seed {seed}: valid {metric} {val:.3f} | test {metric} {test_m:.3f} | {ep} epochs")
    rows.append({"dataset": name, "metric": metric, "test_mean": np.mean(tests), "test_std": np.std(tests)})
results = pd.DataFrame(rows)
results.to_csv("gin_results.csv", index=False)
results

## 7. Tables for the report

In [ ]:
print("Table 1 — dataset and fragment statistics")
print(stats.round(2).to_string(index=False))
print("\nTable 2 — Fragment-GIN baseline (scaffold split, 3 seeds)")
for r in rows:
    print(f"{r['dataset']} & Fragment-GIN & {r['metric']} & {r['test_mean']:.3f} $\\pm$ {r['test_std']:.3f} \\\\")
print(f"\nSaved in {OUT}: fragment_stats.csv, gin_results.csv, figures/fragment_hist.pdf, frag_graphs/, checkpoints/")

**What to put in the report**
- Table 1 from `fragment_stats.csv` (molecules, mean ± std fragments, max, % ≤ 12, % single-fragment).
- The histogram as a figure.
- The Fragment-GIN row as the first entry of the results table, with GSPELL / GNNExplainer rows as placeholders.
- Note in Method: splits are Bemis–Murcko scaffold splits computed locally (80/10/10); counter-ions removed; molecules with no BRICS bonds become single-node graphs.